In [1]:
import sys, os
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

Enabled rmm statistics


In [2]:
%LoadCheckpoint /home/colinc/code/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/rewritten_cpu/o4_mini_high/checkpoints/post_cell_20_try_0.pickle

trying: ['responses_df_2022_cell10', 'responses_df_2022']
me:  16
me:  16
trying: ['responses_df_2022_cell10', 'responses_df_2022']
me:  16
me:  16
trying: ['title_for_x_axis_cell29', 'title_for_x_axis_cell20', 'title_for_x_axis', 'title_for_x_axis_cell22']
me:  17
me:  8
me:  6
me:  10
trying: ['title_for_x_axis_cell29', 'title_for_x_axis_cell20', 'title_for_x_axis', 'title_for_x_axis_cell22']
me:  17
me:  8
me:  6
me:  10
trying: ['title_for_x_axis_cell22', 'title_for_x_axis_cell29', 'title_for_x_axis_cell20', 'title_for_x_axis']
me:  10
me:  17
me:  8
me:  6
trying: ['title_for_x_axis_cell22', 'title_for_x_axis_cell29', 'title_for_x_axis_cell20', 'title_for_x_axis']
me:  10
me:  17
me:  8
me:  6
trying: ['percentages']
me:  5
trying: ['warnings']
me:  0
trying: ['go']
me:  0
trying: ['px']


me:  0
trying: ['sns']
me:  0
trying: ['np']
me:  0
trying: ['df']
me:  14
trying: ['df_cell26']
me:  14
trying: ['percentages_cell19']
me:  7
trying: ['question_of_interest_alternate']
me:  14
trying: ['question_of_interest_cell19']
me:  7
trying: ['learning_platform_df_combined_counts']
me:  14
trying: ['count_then_return_percent_19']
me:  7
trying: ['learning_platform_df_combined_percentages']
me:  14
trying: ['add_year_column_to_dataframes_26']
me:  14
trying: ['grab_subset_of_data_26']
me:  14
trying: ['convert_df_of_counts_to_percentages_26']
me:  14
trying: ['question_of_interest_cell26']
me:  14
trying: ['learning_platform_df_combined']
me:  14
trying: ['combine_subset_of_data_from_multiple_years_for_multiple_choice_multiple_response_questions_26']
me:  14
trying: ['learning_platform_df_combined_percentages_cell26']
me:  14
trying: ['age_df_combined']
me:  8
trying: ['age_df_combined_cell22']
me:  10
trying: ['responses_in_order_cell32']
me:  20
trying: ['count_then_return_perc

me:  17
trying: ['percentages_cell21']
me:  9
trying: ['question_of_interest_cell21']
me:  9
trying: ['count_then_return_percent_21']
me:  9
trying: ['responses_in_order_cell21']
me:  9
trying: ['pd']
me:  0
trying: ['glob']
me:  0
trying: ['responses_df_2018']
me:  1
trying: ['percentages_cell32']
me:  20
trying: ['responses_df_2019']
me:  1
trying: ['question_of_interest_cell32']
me:  20
trying: ['replace_hyphen_with_en_dash']
me:  2
trying: ['s']
me:  20
trying: ['count_then_return_percent_18']
me:  6
trying: ['question_name_alternate_cell18']
me:  6
trying: ['combine_subset_of_data_from_multiple_years_18']
me:  6
trying: ['question_name_alternate']
me:  6
trying: ['subset_of_countries']
me:  6
trying: ['percentages_per_country_df']
me:  4
trying: ['create_dataframe_of_counts_16']
me:  4
trying: ['question_of_interest_cell18']
me:  6
trying: ['country_df_combined']
me:  6
trying: ['country_df_combined_cell18']
me:  6
trying: ['add_year_column_to_dataframes_18']
me:  6
trying: ['ques

In [3]:
%%RecordEvent
%%time
### cell 21 ###

question_of_interest_cell33 = 'For how many years have you been writing code and/or programming?'

# Standardize column names and response categories
responses_df_2018_cell10.rename(
    columns={'How long have you been writing code to analyze data?': question_of_interest_cell33},
    inplace=True
)
mapping_2018 = {
    '< 1 year': '< 1 years',
    'I have never written code but I want to learn': 'I have never written code',
    'I have never written code and I do not want to learn': 'I have never written code',
    '1-2 years': '1-3 years',
    '20-30 years': '20+ years',
    '30-40 years': '20+ years',
    '40+ years': '20+ years'
}
responses_df_2018_cell10[question_of_interest_cell33] = (
    responses_df_2018_cell10[question_of_interest_cell33]
    .replace(mapping_2018)
)

responses_df_2019_cell10.rename(
    columns={'How long have you been writing code to analyze data (at work or at school)?': question_of_interest_cell33},
    inplace=True
)
mapping_1_2_to_1_3 = {'1-2 years': '1-3 years'}
# Apply the same 1–2→1–3 years mapping to both 2019 and 2020
for df in (responses_df_2019_cell10, responses_df_2020):
    df[question_of_interest_cell33] = (
        df[question_of_interest_cell33]
        .replace(mapping_1_2_to_1_3)
    )

def combine_subset_of_data_from_multiple_years_33(question, x_axis_title, include_2017=None):
    # Build a single concatenated DataFrame with a 'year' column
    frames = []
    if include_2017 is not None:
        frames.append(responses_df_2017[[question]].assign(year='2017'))
    for year, df in [
        ('2018', responses_df_2018_cell10),
        ('2019', responses_df_2019_cell10),
        ('2020', responses_df_2020),
        ('2021', responses_df_2021),
        ('2022', responses_df_2022_cell10)
    ]:
        frames.append(df[[question]].assign(year=year))
    combined = pd.concat(frames, ignore_index=True)

    # Compute value counts per year and derive percentages
    counts = combined.groupby('year')[question].value_counts(dropna=False)
    totals = combined.groupby('year')[question].count()
    percentages = (counts.div(totals, level='year') * 100).round(1)

    # Format the result
    result = (
        percentages.rename('percentage')
        .reset_index()
        .rename(columns={question: x_axis_title})
    )
    return result

# Combine, sort, and display
title_for_x_axis_cell33 = ''
programming_df_combined = (
    combine_subset_of_data_from_multiple_years_33(
        question_of_interest_cell33,
        title_for_x_axis_cell33
    )
    .sort_values(['year', 'percentage'], ascending=True)
)
programming_df_combined.info()

<class 'pandas.core.frame.DataFrame'>
Index: 39 entries, 7 to 31
Data columns (total 3 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   year        39 non-null     object 
 1               35 non-null     object 
 2   percentage  39 non-null     float64
dtypes: float64(1), object(2)
memory usage: 1.2+ KB
CPU times: user 21.7 ms, sys: 0 ns, total: 21.7 ms
Wall time: 21.7 ms


In [4]:
%Checkpoint /home/colinc/code/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/rewritten_cpu/o4_mini_high/checkpoints/post_cell_21_try_1.pickle

migration speed (bps): 747087100.8094335


---------------------------
variables to migrate:
pd 72
glob 72
file_name_2017 76
percentages 958
age_df_combined 10611
create_dataframe_of_counts_16 144
percentages_per_country_df 4474
responses_df_2018 32983443
responses_df_2019 16501304
count_then_return_percent_20 144
title_for_x_axis_cell20 49
replace_hyphen_with_en_dash 144
combine_subset_of_data_from_multiple_years_20 144
add_year_column_to_dataframes_20 144
question_of_interest_cell20 76
warnings 72
go 72
px 72
question_of_interest_alternate 123
sns 72
np 72
learning_platform_df_combined_counts 1089
learning_platform_df_combined_percentages 1089
add_year_column_to_dataframes_26 144
grab_subset_of_data_26 144
convert_df_of_counts_to_percentages_26 144
question_of_interest_cell26 117
learning_platform_df_combined 5297884
combine_subset_of_data_from_multiple_years_for_multiple_choice_multiple_response_questions_26 144
learning_platform_df_combined_percentages_cell26 849
df_cell26 5761
percentages_cell17 958
responses_in_order 184


In [5]:
%PrintCellInfo opt_cell_exec_info

======= Cell 0 =======
Input variables []
Active variables []
Intermediate variables []
Future variables []
Modified dataframes
======= Cell 1 =======
Input variables []
Active variables ['responses_df_2021', 'responses_df_2022', 'responses_df_2018', 'responses_df_2020', 'responses_df_2019']
Intermediate variables ['factor', 'file_name_2020', 'base_dir_2017', 'base_dir_2021', 'responses_df_2017', 'base_dir_2019', 'base_dir_2018', 'directory_cell8', 'file_name_2019', 'directory', 'file_name_2022', 'file_name_2021', 'base_dir_2020', 'file_name_2017', 'file_name_2018', 'base_dir_2022']
Future variables ['percentages', 'responses_df_2017', 'responses_df_2022_cell10']
Modified dataframes
======= Cell 2 =======
Input variables ['responses_df_2022', 'responses_df_2018', 'responses_df_2019']
Active variables ['responses_df_2019_cell10', 'responses_df_2022', 'responses_df_2022_cell10']
Intermediate variables ['responses_df_2018_cell10']
Future variables ['responses_df_2017', 'responses_df_2018_

In [6]:

with open("/home/colinc/code/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/opt_cell_exec_info_21_try_1.pkl", "wb") as f:
    pickle.dump(opt_cell_exec_info[21], f)


In [7]:
opt_output = Out.get(4)

In [8]:
title_for_x_axis_cell33_opt = title_for_x_axis_cell33
%LoadCheckpoint /home/colinc/code/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/annotated_cpu/checkpoints/post_cell_21.pickle
assert title_for_x_axis_cell33_opt == title_for_x_axis_cell33

import numpy as np
import cudf
from elastic.core.common.pandas import is_type_styler
is_orig_output_pd = isinstance(orig_output, (pd.Series, pd.DataFrame, pd.Index))
is_opt_output_pd = isinstance(opt_output, (pd.Series, pd.DataFrame, pd.Index))
is_orig_output_array = isinstance(orig_output, (cudf.pandas._wrappers.numpy.ndarray, np.ndarray))
is_opt_output_array = isinstance(opt_output, (cudf.pandas._wrappers.numpy.ndarray, np.ndarray))
is_orig_output_styler = is_type_styler(type(orig_output))
is_opt_output_styler = is_type_styler(type(opt_output))
if is_orig_output_styler and is_opt_output_styler:
    assert orig_output.to_html() == opt_output.to_html()
elif is_orig_output_styler:
    assert orig_output.to_html() == opt_output.to_html()
elif is_opt_output_styler:
    assert opt_output.to_html() == orig_output

if is_orig_output_pd and is_opt_output_pd:
    assert orig_output.equals(opt_output)
# TODO(jie): this is a hack.
elif ((is_orig_output_pd or is_opt_output_pd) and (is_orig_output_array or is_opt_output_array)) or (is_orig_output_array and is_opt_output_array):
    assert list(orig_output) == list(opt_output)
else:
    assert orig_output == opt_output


trying: ['responses_df_2022_cell10', 'responses_df_2022']


me:  32
me:  32
trying: ['responses_df_2022_cell10', 'responses_df_2022']


me:  32
me:  32
trying: ['file_name_2017', 'file_name_2018']
me:  2
me:  2
trying: ['file_name_2017', 'file_name_2018']
me:  2
me:  2
trying: ['title_for_x_axis_cell29', 'title_for_x_axis_cell33', 'title_for_x_axis', 'title_for_x_axis_cell22', 'title_for_x_axis_cell20']
me:  34
me:  42
me:  12
me:  20
me:  16
trying: ['title_for_x_axis_cell29', 'title_for_x_axis_cell33', 'title_for_x_axis', 'title_for_x_axis_cell22', 'title_for_x_axis_cell20']
me:  34
me:  42
me:  12
me:  20
me:  16
trying: ['title_for_x_axis_cell29', 'title_for_x_axis_cell33', 'title_for_x_axis', 'title_for_x_axis_cell22', 'title_for_x_axis_cell20']
me:  34
me:  42
me:  12
me:  20
me:  16
trying: ['title_for_x_axis_cell29', 'title_for_x_axis_cell33', 'title_for_x_axis', 'title_for_x_axis_cell22', 'title_for_x_axis_cell20']
me:  34
me:  42
me:  12
me:  20
me:  16
trying: ['question_name', 'question_of_interest_cell18']
me:  12
me:  12
trying: ['question_name', 'question_of_interest_cell18']
me:  12
me:  12
trying: ['re

me:  34
trying: ['responses_df_2017']


me:  34
trying: ['degree_df_combined']
me:  34
trying: ['responses_df_2018']


...

me:  42
trying: ['age_df_combined']
me:  16
trying: ['programming_df_combined']
me:  42
trying: ['responses_df_2019_cell10']


me:  42
trying: ['count_then_return_percent_19']
me:  14
trying: ['question_of_interest_cell19']
me:  14
trying: ['combine_subset_of_data_from_multiple_years_22']
me:  20
trying: ['count_then_return_percent_33']
me:  42
trying: ['percentages_cell19']
me:  14
trying: ['orig_output']
me:  1
trying: ['question_of_interest_cell30']
me:  36
trying: ['count_then_return_percent_30']
me:  36
trying: ['add_year_column_to_dataframes_33']
me:  42
trying: ['responses_df_2020']


me:  42
trying: ['percentages_cell30']
me:  36
trying: ['country_df_combined_cell18']
me:  12
trying: ['country_df_combined']
me:  12
trying: ['add_year_column_to_dataframes_18']
me:  12
trying: ['question_name_alternate_cell18']
me:  12
trying: ['question_of_interest_cell33']
me:  42
trying: ['count_then_return_percent_18']
me:  12
trying: ['subset_of_countries']
me:  12
trying: ['combine_subset_of_data_from_multiple_years_18']
me:  12
trying: ['question_of_interest_cell31']
me:  38
trying: ['question_name_alternate']
me:  12
trying: ['grab_subset_of_data_31']
me:  38
trying: ['online_learning_platforms_df_2022_cell31']
me:  38
trying: ['df_cell26']
me:  28
trying: ['learning_platform_df_combined']
me:  28
trying: ['learning_platform_df_combined_counts']
me:  28
trying: ['learning_platform_df_combined_percentages_cell26']
me:  28
trying: ['add_year_column_to_dataframes_26']
me:  28
trying: ['grab_subset_of_data_26']
me:  28
trying: ['question_of_interest_alternate']
me:  28
trying: ['

me:  24
trying: ['question_of_interest_cell24']
me:  24
trying: ['title_for_y_axis_cell24']
me:  24
trying: ['USA_responses_df_2022']
me:  24
trying: ['age_df_combined_cell22']
me:  20
trying: ['count_then_return_percent_22']
me:  20
trying: ['question_of_interest_cell22']
me:  20
trying: ['add_year_column_to_dataframes_22']
me:  20
trying: ['title_for_x_axis_cell22']
me:  20
trying: ['question_of_interest']
me:  10
trying: ['online_learning_platforms_df_2022']
me:  26
trying: ['responses_in_order']
me:  10
trying: ['grab_subset_of_data_25']
me:  26
trying: ['percentages_cell17']
me:  10
trying: ['question_of_interest_cell25']
me:  26
trying: ['count_then_return_percent_17']
me:  10
trying: ['grab_subset_of_data_27']
me:  30
trying: ['question_of_interest_cell27']
me:  30
trying: ['online_learning_platforms_df_2022_cell27']
me:  30
trying: ['question_of_interest_cell28']
me:  32
trying: ['percentages_cell28']
me:  32
trying: ['count_then_return_percent_28']
me:  32


Declaring variable warnings
Declaring variable go
Declaring variable np
Declaring variable sns
Declaring variable px
Declaring variable glob
Declaring variable pd
Declaring variable orig_output
Declaring variable file_name_2017
Declaring variable file_name_2018
Declaring variable file_name_2017
Declaring variable file_name_2018
Declaring variable load_survey_data
Declaring variable base_dir_2017
Declaring variable directory_cell8
Declaring variable factor
Declaring variable directory
Declaring variable base_dir_2020
Declaring variable base_dir_2022
Declaring variable file_name_2020
Declaring variable file_name_2019
Declaring variable file_name_2021
Declaring variable base_dir_2018
Declaring variable file_name_2022
Declaring variable base_dir_2019
Declaring variable base_dir_2021
Declaring variable responses_df_2018
Declaring variable responses_df_2019
Declaring variable replace_hyphen_with_en_dash
Declaring variable percentages_per_country_df
Declaring variable create_dataframe_of_coun